In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.nn.functional import relu, sigmoid
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [ ]:
np.random.seed(110007)

In [ ]:
transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])

train_dataset = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root="data", train=False, download=True, transform=transform)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
def train(model, epochs=5):

    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model size: {total_params}')
    
    train_losses = []
    
    for e in range(epochs):
        running_loss = 0.0
        for i, data in enumerate(train_loader, 0):
            X, y = data
            optimizer.zero_grad()
            y_hat = model(X)
            loss = criterion(y_hat, y)
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item()
    
        epoch_loss = running_loss / len(train_dataset)
        train_losses.append(epoch_loss)
        
        print(f'[Epoch {e+1}] Loss: {epoch_loss:.4f}')

def test(model):
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data in test_loader:
            X, y = data
            output = model(X)
            _, y_hat = torch.max(output.data, 1)
            total += y.size(0)
            correct += (y_hat == y).sum().item()
    
    print(f'\nAccuracy of the network on the {total} test images: {100 * correct / total} %')

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv = nn.Conv2d(1, 2, kernel_size=5)
        self.pool = nn.MaxPool2d(kernel_size=6)
        self.fc = nn.Linear(2 * 4 * 4, 10)

    def forward(self, x):
        x = self.pool(relu(self.conv(x)))
        x = x.reshape(-1, 2 * 4 * 4)
        x = self.fc(x)
        return x

In [ ]:
model = CNN()
train(model, epochs=5)
test(model)

In [ ]:
np.random.seed(110007)

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc = nn.Linear(784, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        return self.fc(x)

In [ ]:
model = MLP()
train(model, epochs=5)

In [ ]:
test(model)